# Week 3: Multi-View Reconstruction

This notebook implements the **Incremental SfM Pipeline**.
We loop through a sequence of images, adding them one by one to our 3D map using **PnP**.

In [11]:
import cv2
import numpy as np
import open3d as o3d
import os
import sys
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.sfm import SfMMap
from src.reconstruction import get_intrinsic_from_exif, create_point_cloud

In [12]:
# -- 1. Load Image Sequence --

IMAGE_SEQUENCE = [f'photo{i}.jpeg' for i in range(1, 46)]

data_dir = os.path.join(module_path, 'data2')
images = []

print(f"Attempting to load {len(IMAGE_SEQUENCE)} images from: {data_dir}")

for name in IMAGE_SEQUENCE:
    path = os.path.join(data_dir, name)
    img = cv2.imread(path)
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images.append(img_rgb)
        if int(name.replace('photo', '').replace('.jpeg', '')) % 5 == 0:
            print(f"Loaded {name}...")
    else:
        print(f"Warning: Could not load {name} - Check if file exists in data/")

print(f"\nSuccessfully loaded {len(images)} images.")

Attempting to load 45 images from: /Users/maryamrizwan/Documents/GitHub/CS-436---Project/data2
Loaded photo5.jpeg...
Loaded photo10.jpeg...
Loaded photo15.jpeg...
Loaded photo20.jpeg...
Loaded photo25.jpeg...
Loaded photo30.jpeg...
Loaded photo35.jpeg...
Loaded photo40.jpeg...
Loaded photo45.jpeg...

Successfully loaded 45 images.


In [13]:

# -- 2. Initialize Map (Frame 0 & 1) --

first_image_path = os.path.join(data_dir, IMAGE_SEQUENCE[0])

# Estimate K using EXIF data from the file
K = get_intrinsic_from_exif(first_image_path)

print("Intrinsic Matrix K:\n", K)

# Initialize SfM Class
sfm = SfMMap(K)

# Bootstrap with first two images
sfm.initialize(images[0], images[1], lowe_ratio=0.8)

EXIF not found. Using approximation.
Intrinsic Matrix K:
 [[1.28e+03 0.00e+00 6.40e+02]
 [0.00e+00 1.28e+03 4.80e+02]
 [0.00e+00 0.00e+00 1.00e+00]]
Initializing Map with first two images...
Found 2483 keypoints in img1 and 2679 in img2.
Found 2483 initial matches.
Filtered down to 664 good matches using Lowe's ratio test.
Map initialized with 319 points.


In [14]:
# -- 3. Incremental Loop (Frame 2 -> N) --

for i in range(2, len(images)):
    sfm.add_view(images[i])
    
    # Run BA every 5 frames to fix drift
    if i % 5 == 0:
        sfm.refine()

View Added. Inliers: 25, New Points: 346, Total: 665
View Added. Inliers: 17, New Points: 321, Total: 986
View Added. Inliers: 18, New Points: 295, Total: 1281
View Added. Inliers: 68, New Points: 476, Total: 1757
--- Running Bundle Adjustment on 6 cameras and 1757 points ---
   > Setup: 6 cams, 1757 points, 3642 obs.
   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         7.0559e+15                                    2.27e+16    
       1              3         3.7608e+15      3.30e+15       1.38e+01       1.57e+16    
       2              4         1.6569e+08      3.76e+15       2.31e+01       1.46e+09    
       3              9         1.1122e+08      5.45e+07       1.42e+01       4.82e+08    
       4             10         5.2133e+07      5.91e+07       3.62e+01       2.07e+08    
       5             12         3.6706e+07      1.54e+07       2.02e+01       1.49e+08    
       6             14         3.0971e+07 

In [15]:
# -- 4. Visualization --

points = np.array(sfm.points_3d)
colors = np.array(sfm.colors)

print(f"Final Cloud has {len(points)} points.")

# Filter outliers 
mean = np.mean(points, axis=0)
std = np.std(points, axis=0)
mask = (np.abs(points - mean) < 2.5 * std).all(axis=1)

filtered_points = points[mask]
filtered_colors = colors[mask]

# Create Point Cloud
pcd = create_point_cloud(filtered_points, filtered_colors)

# --- VISUALIZE TRAJECTORY ---
camera_centers = []
camera_frustums = []

AXIS_SIZE = 2.0 

for R, t in sfm.poses:
    # 4x4 matrix
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t.flatten()
    
    # Invert for display (Camera -> World)
    T_inv = np.linalg.inv(T)
    center = T_inv[:3, 3]
    camera_centers.append(center)
    
    axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=AXIS_SIZE, origin=[0,0,0])
    axis.transform(T_inv)
    camera_frustums.append(axis)

# Create a LineSet to connect the camera centers (The Trajectory Path)
if len(camera_centers) > 1:
    lines = []
    for i in range(len(camera_centers) - 1):
        lines.append([i, i+1])
    
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(camera_centers)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    line_set.colors = o3d.utility.Vector3dVector([[1, 0, 0] for _ in lines])
    
    camera_frustums.append(line_set)

geometries = [pcd] + camera_frustums

print("Opening Visualization...")
print(f"Drawing {len(filtered_points)} points and {len(camera_centers)} cameras.")
o3d.visualization.draw_geometries(geometries, window_name="Week 3: Incremental SfM")

Final Cloud has 8775 points.
Opening Visualization...
Drawing 8729 points and 35 cameras.
